# 02 — Data Preprocessing & YOLO Annotations

**Read-only w.r.t. the dataset.** Documents the existing, unchanged BDD100K-to-
YOLO conversion pipeline, confirms V2 trains on the exact same split V1 used (so
every V1-vs-V2 comparison in later notebooks is apples-to-apples), and reports a
new annotation-quality flagging pass.

## 1. Conversion pipeline (unchanged, documented not re-run)

`notebooks/convert_to_yolo.py` is the original, still-authoritative source of
`data/bdd100k/yolo_dataset/`. Summary (full evidence in
`reports/p0_diagnostics/README.md`):

1. Matches `images/10k/{train,val,test}` against a **global** index built across
   every `labels/100k/*` subfolder (so 10k-train images whose label happens to
   sit in `labels/100k/val` are still matched correctly -- a known, documented
   BDD100K packaging quirk, not a bug).
2. Groups images by a filename-prefix heuristic (`stem.split('-')[0]`) and
   assigns whole groups atomically to train/val/test, with a `RuntimeError` guard
   against any group overlapping test -- independently confirmed in P0 to be the
   real BDD100K video ID, with zero actual cross-split overlap.
3. Maps BDD100K's 10-class detection schema to this project's 9 classes
   (drops `train`-the-vehicle, ~179 instances total in all of BDD100K --
   negligibly rare, standard practice).

**V2 does not modify this script or re-run the conversion.**

In [1]:
import yaml

from pathlib import Path



YOLO_DS = Path('../../data/bdd100k/yolo_dataset')

data_yaml = yaml.safe_load((YOLO_DS / 'data.yaml').read_text())

print(data_yaml)



for split in ('train', 'val', 'test'):

    n_images = len(list((YOLO_DS / 'images' / split).glob('*.jpg')))

    n_labels = len(list((YOLO_DS / 'labels' / split).glob('*.txt')))

    print(f'{split}: {n_images} images, {n_labels} labels')

{'path': 'C:/Users/wahbi/OneDrive/Bureau/DriveSight_New/data/bdd100k/yolo_dataset', 'train': 'images/train', 'val': 'images/val', 'test': 'images/test', 'names': {0: 'pedestrian', 1: 'rider', 2: 'car', 3: 'truck', 4: 'bus', 5: 'motorcycle', 6: 'bicycle', 7: 'traffic light', 8: 'traffic sign'}}
train: 2744 images, 2744 labels
val: 686 images, 686 labels
test: 830 images, 830 labels

**Confirmed: 2,744 / 686 / 830 train/val/test images**, matching every number
reported across the P0 diagnostics and resolution-ablation rounds exactly. V2
trains on this identical split -- `scripts/drivesight_v2/train_v2.py` points at
this same `data.yaml` with no copy or modification.

## 2. New: annotation-ambiguity flagging pass (read-only, not applied to training)

The P0 error-cause analysis found one concrete case
(`a48909f7-54e4bcff.jpg`) of a `rider`-labeled ground-truth box with no
bicycle/motorcycle visible anywhere in the frame -- a person plainly walking in a
crosswalk. `scripts/drivesight_v2/flag_annotation_ambiguity.py` checks how common
this specific pattern is across the whole dataset.

In [1]:
import subprocess, sys

r = subprocess.run([sys.executable, '../../scripts/drivesight_v2/flag_annotation_ambiguity.py'],

                    capture_output=True, text=True)

print(r.stdout)

Rider-containing images (all splits): 294
Flagged (rider present, no bicycle/motorcycle anywhere in image): 19 (6.5% of rider-containing images)
NOT auto-removed from training data -- candidate list for human review only.
Saved: C:\Users\wahbi\OneDrive\Bureau\DriveSight_New\reports\drivesight_v2\preprocessing\flagged_rider_without_vehicle.csv

In [1]:
import pandas as pd

flagged = pd.read_csv('../../reports/drivesight_v2/preprocessing/flagged_rider_without_vehicle.csv')

display(flagged.head(10))

split        image_stem  rider_count                                            flag
train 07b5af1c-5153aa63            3 rider_present_no_bicycle_or_motorcycle_in_image
train 23dfa637-bed90829            1 rider_present_no_bicycle_or_motorcycle_in_image
train 2fa3d5f3-60a764a1            1 rider_present_no_bicycle_or_motorcycle_in_image
train 381582b9-8de52fd5            3 rider_present_no_bicycle_or_motorcycle_in_image
train 4b4cb9b0-3efa6143            3 rider_present_no_bicycle_or_motorcycle_in_image
train 4e196e95-a47e4edb            1 rider_present_no_bicycle_or_motorcycle_in_image
train 58208859-5316743a            1 rider_present_no_bicycle_or_motorcycle_in_image
train 583fc3b5-0e8f3e55            1 rider_present_no_bicycle_or_motorcycle_in_image
train 626ec015-61ac91c1            1 rider_present_no_bicycle_or_motorcycle_in_image
train 8c7b5aed-4efca596            1 rider_present_no_bicycle_or_motorcycle_in_image

**6.5% of rider-containing images (19 of 294) have no bicycle/motorcycle visible
anywhere in frame.** Non-trivial on an already-thin rider dataset, but not large
enough to be the dominant driver of rider's AP gap on its own. **Decision: not
auto-filtered.** Removing labels without human verification on a dataset this
small carries real risk of discarding genuinely-correct-but-unusual instances
(e.g. a rider who dismounted mid-frame); the flagged list
(`reports/drivesight_v2/preprocessing/flagged_rider_without_vehicle.csv`) is
carried forward as a documented, reviewable candidate list for a future manual
cleaning pass, not applied to V2 training.

## Conclusion

- V2's training/validation/test data is byte-for-byte the same split V1 used --
  every comparison in notebooks 03-07 is on identical data.
- A real but modest (6.5%) rider/vehicle annotation-ambiguity rate is documented,
  not silently ignored, and explicitly deferred rather than acted on blindly.
- No dataset file was created, modified, or deleted by this notebook.